# Session 5 — Analysis: Precision / Dtype (FP32 / FP16 / BF16 / INT8)

## Background

Sessions 1–4 ran exclusively in FP32. Modern accelerators expose lower-precision math units that deliver higher FLOP/s at reduced memory footprint: NVIDIA Tensor Cores accelerate FP16 and BF16 matrix multiplications; TPU MXUs are native BF16. INT8 quantization goes further — 8-bit arithmetic at ~2× the throughput of BF16. This session compares all three reduced-precision paths on both devices.

## Goal

Compare how much throughput FP16, BF16, and INT8 add on the GPU vs how much BF16 and INT8 add on the TPU, across six batch sizes. Determine whether the MXU's native BF16 support produces a larger speedup than the GPU's Tensor Core path, and quantify VRAM savings on the GPU side.

## Question

Which device benefits more from reduced precision, and how does that interact with batch size?

---

**Prerequisites:** Run `01-benchmark-gpu.ipynb` and `02-benchmark-tpu.ipynb` first to populate `results/gpu_dtype_sweep.json` and `results/tpu_dtype_sweep.json`.

In [ ]:
import json, os

def load_results(results_dir="results"):
    out = {}
    for fname in ["gpu_dtype_sweep.json", "tpu_dtype_sweep.json"]:
        path = os.path.join(results_dir, fname)
        if not os.path.exists(path):
            print(f"Missing: {path} — run the corresponding benchmark notebook first.")
            continue
        with open(path) as f:
            d = json.load(f)
        hw = d["hardware"]
        out[hw] = d
    return out

data = load_results("results")

for hw, d in data.items():
    print(f"{hw} ({d['device_name']}) — dtypes: {d['dtypes_tested']}")
    for dtype_str, batches in d["results"].items():
        row = f"  {dtype_str.upper():<6}: "
        for bs, r in batches.items():
            if r and isinstance(r, dict):
                row += f"  batch={bs}: {r['throughput']:,.0f}  "
            elif r:
                row += f"  batch={bs}: {r:,.0f}  "
            else:
                row += f"  batch={bs}: OOM  "
        print(row)

GPU (NVIDIA L4) — dtypes: ['fp32', 'fp16', 'bf16']
TPU (TPU) — dtypes: ['fp32', 'bf16']


In [ ]:
BATCH_SIZES = [32, 64, 128, 256, 512, 1024]

def get_tput(d, dtype_str, bs):
    r = d["results"].get(dtype_str, {}).get(str(bs))
    if r is None:
        return None
    if isinstance(r, dict):
        return r.get("throughput")
    return r

for hw, d in data.items():
    dtypes = d["dtypes_tested"]
    print(f"\n{'═'*70}")
    print(f"  {hw} ({d['device_name']})")
    print(f"{'═'*70}")
    header = f"  {'batch':<8}" + "".join(f" {dt.upper():>16}" for dt in dtypes)
    if "bf16" in dtypes and "fp32" in dtypes:
        header += f"  {'BF16/FP32':>12}"
    if "int8" in dtypes:
        header += f"  {'INT8/FP32':>12}"
    print(header)
    print(f"  {'-'*8}" + "".join(f" {'-'*16}" for _ in dtypes) +
          ("  " + "-"*12 if "bf16" in dtypes and "fp32" in dtypes else "") +
          ("  " + "-"*12 if "int8" in dtypes else ""))
    for bs in BATCH_SIZES:
        row = f"  {bs:<8}"
        fp32_tput = get_tput(d, "fp32", bs)
        bf16_tput = get_tput(d, "bf16", bs)
        int8_tput = get_tput(d, "int8", bs)
        for dt in dtypes:
            t = get_tput(d, dt, bs)
            row += f" {f'{t:,.0f}' if t else 'OOM':>16}"
        if fp32_tput and bf16_tput:
            row += f"  {bf16_tput / fp32_tput:>12.3f}×"
        if "int8" in dtypes and fp32_tput and int8_tput:
            row += f"  {int8_tput / fp32_tput:>12.3f}×"
        print(row)

# GPU VRAM summary
if "GPU" in data:
    print(f"\n{'─'*50}")
    print("  GPU peak VRAM (MB) at batch=256")
    print(f"{'─'*50}")
    for dt in data["GPU"]["dtypes_tested"]:
        r = data["GPU"]["results"].get(dt, {}).get("256")
        if r and isinstance(r, dict) and r.get("peak_vram_mb"):
            print(f"  {dt.upper():<6}: {r['peak_vram_mb']:,.0f} MB")


══════════════════════════════════════════════════════════════════════
  GPU (NVIDIA L4)
══════════════════════════════════════════════════════════════════════
  batch                FP32             FP16             BF16     BF16/FP32
  -------- ---------------- ---------------- ----------------  ------------
  32                  2,903            7,206            7,208         2.483×
  64                  2,705            6,909            6,902         2.552×
  128                 2,637            5,916            5,730         2.173×
  256                 2,583            5,880            5,643         2.184×
  512                 2,579            5,822            5,591         2.168×
  1024                2,524            5,497            5,259         2.084×


In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import os

COLORS = {
    "fp32": "#888888",
    "fp16": "#ff9900",
    "bf16": "#4285F4",
    "int8": "#e74c3c",
}
LABELS = {"fp32": "FP32", "fp16": "FP16", "bf16": "BF16", "int8": "INT8"}

hw_order = [hw for hw in ["GPU", "TPU"] if hw in data]
fig, axes = plt.subplots(1, len(hw_order), figsize=(13, 5), sharey=False)
if len(hw_order) == 1:
    axes = [axes]

for ax, hw in zip(axes, hw_order):
    d = data[hw]
    for dt in d["dtypes_tested"]:
        tputs = [get_tput(d, dt, bs) for bs in BATCH_SIZES]
        valid_bs   = [b for b, t in zip(BATCH_SIZES, tputs) if t is not None]
        valid_tput = [t for t in tputs if t is not None]
        ax.plot(valid_bs, valid_tput, marker="o", label=LABELS[dt],
                color=COLORS[dt], linewidth=2)
    ax.set_xscale("log", base=2)
    ax.set_xticks(BATCH_SIZES)
    ax.set_xticklabels(BATCH_SIZES)
    ax.set_xlabel("Batch size", fontsize=11)
    ax.set_ylabel("Throughput (samples / second)", fontsize=11)
    ax.set_title(f"{hw} ({d['device_name']})", fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))

fig.suptitle("Dtype Precision Sweep — Throughput vs Batch Size", fontsize=13)
plt.tight_layout()

os.makedirs("../docs/assets", exist_ok=True)
plt.savefig("../docs/assets/session_5_chart_throughput.png", dpi=150)
print("Saved session_5_chart_throughput.png")

# ── Chart 2: BF16 speedup ratio vs batch size ─────────────────────────────────
fig2, ax2 = plt.subplots(figsize=(8, 4))
HW_COLORS = {"GPU": "#76b900", "TPU": "#4285F4"}

for hw in hw_order:
    d = data[hw]
    ratios, valid_bs = [], []
    for bs in BATCH_SIZES:
        fp32_t = get_tput(d, "fp32", bs)
        bf16_t = get_tput(d, "bf16", bs)
        if fp32_t and bf16_t:
            ratios.append(bf16_t / fp32_t)
            valid_bs.append(bs)
    ax2.plot(valid_bs, ratios, marker="o", label=hw, color=HW_COLORS[hw], linewidth=2)

ax2.axhline(1.0, color="gray", linestyle="--", linewidth=1, label="FP32 baseline (1.0×)")
ax2.set_xscale("log", base=2)
ax2.set_xticks(BATCH_SIZES)
ax2.set_xticklabels(BATCH_SIZES)
ax2.set_xlabel("Batch size", fontsize=11)
ax2.set_ylabel("BF16 / FP32 throughput ratio", fontsize=11)
ax2.set_title("BF16 Speedup vs FP32 — GPU vs TPU", fontsize=12)
ax2.legend(fontsize=10)
ax2.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.savefig("../docs/assets/session_5_chart_bf16_speedup.png", dpi=150)
print("Saved session_5_chart_bf16_speedup.png")

Saved session_5_chart_throughput.png
Saved session_5_chart_bf16_speedup.png


In [ ]:
if "GPU" in data:
    fig, ax = plt.subplots(figsize=(8, 4))

    for dt in data["GPU"]["dtypes_tested"]:
        vrams, valid_bs = [], []
        for bs in BATCH_SIZES:
            r = data["GPU"]["results"].get(dt, {}).get(str(bs))
            if r and isinstance(r, dict) and r.get("peak_vram_mb"):
                vrams.append(r["peak_vram_mb"])
                valid_bs.append(bs)
        if vrams:
            ax.plot(valid_bs, vrams, marker="o", label=dt.upper(),
                    color=COLORS[dt], linewidth=2, markersize=6)

    ax.set_xscale("log", base=2)
    ax.set_xticks(BATCH_SIZES)
    ax.set_xticklabels(BATCH_SIZES)
    ax.set_xlabel("Batch size", fontsize=11)
    ax.set_ylabel("Peak VRAM (MB)", fontsize=11)
    ax.set_title("GPU Peak VRAM — FP16/BF16 halves memory at every batch size", fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(True, linestyle="--", alpha=0.4)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
    plt.tight_layout()

    plt.savefig("../docs/assets/session_5_chart_vram.png", dpi=150)
    print("Saved session_5_chart_vram.png")

Saved session_5_chart_vram.png


## Interpreting the results

| Observation | What it means |
|---|---|
| GPU FP16/BF16 speedup | Tensor Cores take over matrix multiplications; memory footprint halves, allowing larger effective batch |
| TPU BF16 speedup | MXU is natively BF16; FP32 path is emulated via wider arithmetic — BF16 is the fast path |
| INT8 speedup | 8-bit arithmetic doubles the MXU TOPS; memory footprint halves again vs BF16 |
| GPU VRAM ~2× lower in BF16 | Activations and gradients halve in size; enables larger batch at the same memory budget |

## When to use each precision

- **FP32:** Numerically sensitive reductions, loss accumulation, optimizer state (AdamW) — handled automatically by `torch.autocast`.
- **BF16 (default for training):** Drop-in replacement for FP32; no gradient scaler needed. Both devices have hardware-accelerated BF16 paths. Switch immediately if you haven't.
- **FP16 (GPU only):** Requires a GradScaler to prevent gradient underflow. BF16 is a strict improvement on modern hardware.
- **INT8 (inference):** Maximum throughput for serving. Not suitable for training weight updates — gradients must remain in FP32/BF16.

## Decision guidance

- If you're not already using BF16, switch immediately — it's free throughput on both devices.
- The TPU's BF16 speedup is larger in absolute terms (native MXU path vs emulated FP32).
- The GPU VRAM reduction from BF16 directly enables larger batches, compounding the throughput gain.
- Use INT8 for inference workloads where you can afford post-training quantization.